# EY Water Quality Challenge EDA

This notebook explores the merged training dataset, target distributions, missingness, geographic coverage, and a few feature-target relationships.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from data_loading import build_model_tables
from features import TARGETS, add_features

sns.set_theme(style='whitegrid')

In [ ]:
train_full, test_full = build_model_tables()
train_df = add_features(train_full)

print(train_df.shape)
train_df.head()

## Missingness overview

In [ ]:
missing_pct = train_df.isna().mean().sort_values(ascending=False).head(20) * 100
plt.figure(figsize=(10, 6))
sns.barplot(x=missing_pct.values, y=missing_pct.index, color='#dd6b20')
plt.title('Top 20 Missing Features')
plt.xlabel('Missing Percentage')
plt.ylabel('')
plt.show()

## Target distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, target in zip(axes, TARGETS):
    sns.histplot(train_df[target], kde=True, ax=ax, color='#2b6cb0')
    ax.set_title(target)
plt.tight_layout()
plt.show()

## Geographic sampling pattern

In [ ]:
plt.figure(figsize=(7, 6))
sns.scatterplot(
    data=train_df,
    x='Longitude',
    y='Latitude',
    hue='Electrical Conductance',
    palette='viridis',
    s=20,
    alpha=0.7,
)
plt.title('Sampling Locations Colored by Electrical Conductance')
plt.show()

## Correlation heatmap

In [ ]:
corr_cols = [col for col in ['nir', 'green', 'swir16', 'pet', 'NDMI', 'MNDWI'] if col in train_df.columns]
corr = train_df[corr_cols + TARGETS].corr(numeric_only=True)
plt.figure(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Feature and Target Correlations')
plt.show()

## Quick feature-target diagnostics

In [ ]:
candidate_features = [feature for feature in ['pet', 'nir', 'green', 'swir16', 'NDWI_green', 'NDWI_swir'] if feature in train_df.columns]
for feature in candidate_features[:4]:
    plt.figure(figsize=(6, 4))
    sns.scatterplot(data=train_df, x=feature, y='Dissolved Reactive Phosphorus', alpha=0.5, s=18)
    plt.title(f'{feature} vs Dissolved Reactive Phosphorus')
    plt.show()